# 03. Bracken Community Structure

This notebook uses Bracken bacterial species counts for the main compositional analyses.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, SVG, display

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import workflow_core as wc

context, base_data, base, advanced = wc.bootstrap_notebook()


## Build Pairwise Community Distances


In [ ]:
analysis_context = wc.base_analysis_context(context)
metadata = base_data["metadata"]
qc = base_data["qc"]
species_bac = base_data["species_bac"]

community_samples = qc.index[qc["community_qc_pass"]].tolist()
rel_abundance = base.community_relative_abundance(species_bac, community_samples)
pairwise = base.summarize_pairwise_distances(rel_abundance, metadata)
summary = base.make_distance_figure(pairwise, analysis_context)

wc.save_table(pairwise, wc.table_path(context, 4, "pairwise_distances"))
wc.save_table(summary, wc.table_path(context, 5, "pairwise_distance_summary"))


## Review Numbered Outputs


In [ ]:
pairwise = pd.read_csv(wc.table_path(context, 4, "pairwise_distances"), sep="\t")
summary = pd.read_csv(wc.table_path(context, 5, "pairwise_distance_summary"), sep="\t")

display(SVG(filename=str(wc.figure_path(context, 2, "pairwise_distance"))))
display(summary)

same_visit = summary.loc[summary["comparison_group"] == "Same patient, same batch date"].iloc[0]
revisit = summary.loc[summary["comparison_group"] == "Same patient, different batch date"].iloc[0]
different = summary.loc[summary["comparison_group"] == "Different patient"].iloc[0]
summary_lines = [
    f"- Positive result: same-patient same-batch-date swabs were much closer than unrelated swabs (median Bray-Curtis {same_visit['median']:.3f} vs {different['median']:.3f}).",
    f"- Negative result: same-patient different-batch-date swabs were nearly as dissimilar as unrelated swabs (median {revisit['median']:.3f}).",
    "- Interpretation: descriptive patient-date grouping is informative, but later models treat absolute date as technical batch rather than biology.",
]
display(Markdown("## Working Interpretation\n" + "\n".join(summary_lines)))
